In [0]:
dbutils.widgets.text("target_environment", "")
environment = dbutils.widgets.get("target_environment")
from pyspark.sql.functions import sha2, concat_ws, coalesce, lit, col
environment = 'dev'
name = "analytics_dq_"
catalog = f"{name}{environment}"

##############################################################################################
####################################     USER INPUTS     #####################################
##############################################################################################

# ================= 1. DIMENSIONS & TYPES =================
user_rule_types = [
    {"rule_type": "technical", "description": "DQ", "added_by": "Demo"},
    {"rule_type": "business", "description": "Business Rules", "added_by": "Demo"}
]

user_rule_dimensions = [
    {"rule_dimension": "validity", "description": "Business logic", "added_by": "Demo"},
    {"rule_dimension": "uniqueness", "description": "Duplicates", "added_by": "Demo"}
]

user_projects = [
    {"project": "demo_project", "project_description": "Demo", "added_by": "Demo"}
]

# ================= 2. POLICIES =================
user_values_run_policy = [
    {"criticality": "high", "category": "error", "quarantine": True, "added_by": "Demo"},
    {"criticality": "low", "category": "warning", "quarantine": False, "added_by": "Demo"}
]

# ================= 3. APPLY_AT =================
user_values_apply_at = [
    {"apply_at": "gold", "is_project_specific": False, "added_by": "Demo"}
]

# ================= 4. RULE TEMPLATES =================
user_values_rule_templates = [
    {
        "name": "is_not_null", "description": "value is not null.", "rule_type_nk": "technical", 
        "rule_dimension_nk": "validity", "scope": "column", "is_reusable": True, 
        "engine_type": "dqx", "statement": "is_not_null", "added_by": "Demo"
    },
    {
        "name": "primary_key_violation", "description": "Primary Key Violation", "rule_type_nk": "technical", 
        "rule_dimension_nk": "uniqueness", "scope": "table", "is_reusable": True, 
        "engine_type": "sql", "statement": "select ${i:columns} from ${table} where 1=1 group by ${i:columns} having count(*) > 1", "added_by": "Demo"
    },
    {
        "name": "check_thresholds_dynamic", "description": "Value must be between min and max", "rule_type_nk": "business", 
        "rule_dimension_nk": "validity", "scope": "column", "is_reusable": True, 
        "engine_type": "sql", "statement": "SELECT * FROM ${table} WHERE ${i:column} < ${v:min_val} OR ${i:column} > ${v:max_val}", "added_by": "Demo"
    }
]

# ================= 5. TABLE REGISTRATION =================
user_values = [
    {"environment": "dev", "catalog": "analytics_dq_dev", "schema": "metadata", "table": "test_trips", "layer": "gold", "primary_keys": ["trip_id"], "filter_field": "tpep_dropoff_datetime", "filter_field_type": "timestamp", "added_by": "Demo"},
    {"environment": "dev", "catalog": "analytics_dq_dev", "schema": "metadata", "table": "test_trips_clean", "layer": "gold", "primary_keys": ["trip_id"], "filter_field": "tpep_dropoff_datetime", "filter_field_type": "timestamp", "added_by": "Demo"},
    {"environment": "dev", "catalog": "analytics_dq_dev", "schema": "metadata", "table": "test_trips_warning", "layer": "gold", "primary_keys": ["trip_id"], "filter_field": "tpep_dropoff_datetime", "filter_field_type": "timestamp", "added_by": "Demo"}
]

# ================= 6. RULE ASSIGNMENTS =================
user_values_rule_assignment = [
    # -----> ASSIGNMENTS FOR THE DIRTY TABLE (test_trips)
    {"table_nk": "analytics_dq_dev.metadata.test_trips", "template_nk": "is_not_null", "policy_nk": "error.high.quarantine", "apply_at_nk": "gold.project_agnostic", "parameters_identifiers": '{"column": "fare_amount"}', "parameters_values": "", "project_nk": "demo_project", "added_by": "Demo"},
    {"table_nk": "analytics_dq_dev.metadata.test_trips", "template_nk": "is_not_null", "policy_nk": "warning.low.no_quarantine", "apply_at_nk": "gold.project_agnostic", "parameters_identifiers": '{"column": "dropoff_zip"}', "parameters_values": "", "project_nk": "demo_project", "added_by": "Demo"},
    {"table_nk": "analytics_dq_dev.metadata.test_trips", "template_nk": "primary_key_violation", "policy_nk": "error.high.quarantine", "apply_at_nk": "gold.project_agnostic", "parameters_identifiers": '{"columns": ["trip_id"]}', "parameters_values": "", "project_nk": "demo_project", "added_by": "Demo"},
    {"table_nk": "analytics_dq_dev.metadata.test_trips", "template_nk": "check_thresholds_dynamic", "policy_nk": "error.high.quarantine", "apply_at_nk": "gold.project_agnostic", "parameters_identifiers": '{"column": "trip_distance"}', "parameters_values": '{"min_val": "0", "max_val": "100"}', "project_nk": "demo_project", "added_by": "Demo"},
    
    # -----> ASSIGNMENTS FOR THE CLEAN TABLE (test_trips_clean)
    {"table_nk": "analytics_dq_dev.metadata.test_trips_clean", "template_nk": "is_not_null", "policy_nk": "error.high.quarantine", "apply_at_nk": "gold.project_agnostic", "parameters_identifiers": '{"column": "fare_amount"}', "parameters_values": "", "project_nk": "demo_project", "added_by": "Demo"},
    {"table_nk": "analytics_dq_dev.metadata.test_trips_clean", "template_nk": "check_thresholds_dynamic", "policy_nk": "error.high.quarantine", "apply_at_nk": "gold.project_agnostic", "parameters_identifiers": '{"column": "trip_distance"}', "parameters_values": '{"min_val": "0", "max_val": "100"}', "project_nk": "demo_project", "added_by": "Demo"},
    
    # -----> ASSIGNMENTS FOR THE WARNING TABLE (test_trips_warning)
    {"table_nk": "analytics_dq_dev.metadata.test_trips_warning", "template_nk": "is_not_null", "policy_nk": "warning.low.no_quarantine", "apply_at_nk": "gold.project_agnostic", "parameters_identifiers": '{"column": "dropoff_zip"}', "parameters_values": "", "project_nk": "demo_project", "added_by": "Demo"}
]
##############################################################################################

schema_metadata = "metadata"

##### rule_type ######

df = spark.createDataFrame(user_rule_types)
df.createOrReplaceTempView("rule_type_inputs")

sql_prepared = f"""
CREATE OR REPLACE TEMP VIEW prepared_rule_type AS
SELECT
    src.rule_type AS rule_type_nk,
    src.rule_type,
    src.description,
    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(src.description AS string), '_NULL_')
        ), 256
    ) AS new_hash
FROM rule_type_inputs src
"""

sql_close = f"""
MERGE INTO {catalog}.{schema_metadata}.rule_type AS t
USING prepared_rule_type AS s
ON t.rule_type_nk = s.rule_type_nk
AND t.is_active = TRUE
AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.description AS string), '_NULL_')
            ), 256
        ) <> s.new_hash

WHEN MATCHED THEN 
UPDATE SET
    t.is_active = FALSE,
    t.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing = f"""
UPDATE {catalog}.{schema_metadata}.rule_type t 
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_rule_type s
    WHERE s.rule_type_nk = t.rule_type_nk
    )
"""

sql_insert = f"""
INSERT INTO {catalog}.{schema_metadata}.rule_type (
    rule_type_nk,
    rule_type,
    description,
    added_by,
    is_active,
    valid_from,
    valid_to
)
SELECT
    s.rule_type_nk,
    s.rule_type,
    s.description,
    s.added_by,
    true,
    current_timestamp(),
    NULL
FROM prepared_rule_type s
LEFT ANTI JOIN {catalog}.{schema_metadata}.rule_type t
    ON  t.rule_type_nk = s.rule_type_nk
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.description AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared)
    spark.sql(sql_close)
    spark.sql(sql_close_missing)
    spark.sql(sql_insert)
    print("SCD2 rule_type update completed successfully.")
except Exception as e:
    print("Error during SCD2 update:", e)

##### rule_dimension ######

df = spark.createDataFrame(user_rule_dimensions)
df.createOrReplaceTempView("rule_dimension_inputs")

sql_prepared = f"""
CREATE OR REPLACE TEMP VIEW prepared_rule_dimension AS
SELECT
    src.rule_dimension AS rule_dimension_nk,
    src.rule_dimension,
    src.description,
    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(src.description AS string), '_NULL_')
        ), 256
    ) AS new_hash
FROM rule_dimension_inputs src
"""

sql_close = f"""
MERGE INTO {catalog}.{schema_metadata}.rule_dimension AS t
USING prepared_rule_dimension AS s
ON t.rule_dimension_nk = s.rule_dimension_nk
AND t.is_active = TRUE
AND sha2(
        concat_ws(
            '||',
            coalesce(cast(t.description AS string), '_NULL_')
        ), 256
    ) <> s.new_hash

WHEN MATCHED THEN
UPDATE SET
    t.is_active = FALSE,
    t.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing = f"""
UPDATE {catalog}.{schema_metadata}.rule_dimension t
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_rule_dimension s
    WHERE s.rule_dimension_nk = t.rule_dimension_nk
)
"""

sql_insert = f"""
INSERT INTO {catalog}.{schema_metadata}.rule_dimension (
    rule_dimension_nk,
    rule_dimension,
    description,
    added_by,
    is_active,
    valid_from,
    valid_to
)
SELECT
    s.rule_dimension_nk,
    s.rule_dimension,
    s.description,
    s.added_by,
    true,
    current_timestamp(),
    NULL
FROM prepared_rule_dimension s
LEFT ANTI JOIN {catalog}.{schema_metadata}.rule_dimension t
    ON  t.rule_dimension_nk = s.rule_dimension_nk
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.description AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared)
    spark.sql(sql_close)
    spark.sql(sql_close_missing)
    spark.sql(sql_insert)
    print("SCD2 rule_dimension update completed successfully.")
except Exception as e:
    print("Error during SCD2 update:", e)

##### table ######

df = spark.createDataFrame(user_values)
df.createOrReplaceTempView("table_inputs")

sql_prepared = f"""
CREATE OR REPLACE TEMP VIEW prepared_table AS
SELECT
    CONCAT(src.catalog, '.', src.schema, '.', src.`table`) AS table_nk,
    src.environment,
    src.catalog,
    src.schema,
    src.`table`,
    src.layer,
    src.primary_keys,
    src.filter_field,
    src.filter_field_type,
    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(src.primary_keys AS string), '_NULL_'),
            coalesce(cast(src.filter_field AS string), '_NULL_'),
            coalesce(cast(src.filter_field_type AS string), '_NULL_')
        ), 256
    ) AS new_hash
FROM table_inputs src
"""

sql_close = f"""
MERGE INTO {catalog}.{schema_metadata}.`table` AS t
USING prepared_table AS s
ON t.table_nk = s.table_nk
AND t.is_active = TRUE
AND sha2(
        concat_ws(
            '||',
            coalesce(cast(t.primary_key AS string), '_NULL_'),
            coalesce(cast(t.filter_field AS string), '_NULL_'),
            coalesce(cast(t.filter_field_type AS string), '_NULL_')
        ), 256
    ) <> s.new_hash

WHEN MATCHED THEN
UPDATE SET
    t.is_active = FALSE,
    t.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing = f"""
UPDATE {catalog}.{schema_metadata}.`table` t
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_table s
    WHERE s.table_nk = t.table_nk
)
"""

sql_insert = f"""
INSERT INTO {catalog}.{schema_metadata}.`table` (
    table_nk,
    environment,
    catalog,
    schema,
    `table`,
    layer,
    primary_key,
    filter_field,
    filter_field_type,
    added_by,
    is_active,
    valid_from,
    valid_to
)
SELECT
    s.table_nk,
    s.environment,
    s.catalog,
    s.schema,
    s.`table`,
    s.layer,
    s.primary_keys,
    s.filter_field,
    s.filter_field_type,
    s.added_by,
    true,
    current_timestamp(),
    NULL
FROM prepared_table s
LEFT ANTI JOIN {catalog}.{schema_metadata}.`table` t
    ON  t.table_nk = s.table_nk
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.primary_key AS string), '_NULL_'),
                coalesce(cast(t.filter_field AS string), '_NULL_'),
                coalesce(cast(t.filter_field_type AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared)
    spark.sql(sql_close)
    spark.sql(sql_close_missing)
    spark.sql(sql_insert)
    print("SCD2 table update completed successfully.")
except Exception as e:
    print("Error during SCD2 update:", e)


##### project ######

df = spark.createDataFrame(user_projects)
df.createOrReplaceTempView("project_inputs")

sql_prepared = f"""
CREATE OR REPLACE TEMP VIEW prepared_project AS
SELECT
    src.project AS project_nk,
    src.project,
    src.project_description,
    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(src.project_description AS string), '_NULL_')
        ), 256
    ) AS new_hash
FROM project_inputs src
"""

sql_close = f"""
MERGE INTO {catalog}.{schema_metadata}.project AS t
USING prepared_project AS s
ON t.project_nk = s.project_nk
AND t.is_active = TRUE
AND sha2(
        concat_ws(
            '||',
            coalesce(cast(t.project_description AS string), '_NULL_')
        ), 256
    ) <> s.new_hash

WHEN MATCHED THEN
UPDATE SET
    t.is_active = FALSE,
    t.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing = f"""
UPDATE {catalog}.{schema_metadata}.project t
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_project s
    WHERE s.project_nk = t.project_nk
)
"""

sql_insert = f"""
INSERT INTO {catalog}.{schema_metadata}.project (
    project_nk,
    project,
    project_description,
    added_by,
    is_active,
    valid_from,
    valid_to
)
SELECT
    s.project_nk,
    s.project,
    s.project_description,
    s.added_by,
    true,
    current_timestamp(),
    NULL
FROM prepared_project s
LEFT ANTI JOIN {catalog}.{schema_metadata}.project t
    ON  t.project_nk = s.project_nk
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.project_description AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared)
    spark.sql(sql_close)
    spark.sql(sql_close_missing)
    spark.sql(sql_insert)
    print("SCD2 project update completed successfully.")
except Exception as e:
    print("Error during SCD2 update:", e)

schema = "configuration"

##### run_policy ######

df_policy = spark.createDataFrame(user_values_run_policy)
df_policy.createOrReplaceTempView("policy_inputs")

sql_prepared_policy = f"""
CREATE OR REPLACE TEMP VIEW prepared_run_policy AS
SELECT
    CONCAT(
        src.category, '.', 
        src.criticality, '.', 
        CASE WHEN src.quarantine THEN 'quarantine' ELSE 'no_quarantine' END
    ) AS policy_nk,

    src.category,
    src.criticality,
    src.quarantine,
    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(src.category AS string), '_NULL_'),
            coalesce(cast(src.criticality AS string), '_NULL_'),
            coalesce(cast(src.quarantine AS string), '_NULL_')
        ), 256
    ) AS new_hash
FROM policy_inputs src
"""

sql_close_policy = f"""
MERGE INTO {catalog}.{schema}.run_policy AS t
USING prepared_run_policy AS s
ON t.policy_nk = s.policy_nk
AND t.is_active = TRUE
AND sha2(
        concat_ws(
            '||',
            coalesce(cast(t.category AS string), '_NULL_'),
            coalesce(cast(t.criticality AS string), '_NULL_'),
            coalesce(cast(t.quarantine AS string), '_NULL_')
        ), 256
    ) <> s.new_hash

WHEN MATCHED THEN
UPDATE SET
    t.is_active = FALSE,
    t.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing_policy = f"""
UPDATE {catalog}.{schema}.run_policy t
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_run_policy s
    WHERE s.policy_nk = t.policy_nk
)
"""

sql_insert_policy = f"""
INSERT INTO {catalog}.{schema}.run_policy (
    policy_nk,
    category,
    criticality,
    quarantine,
    added_by,
    is_active,
    valid_from,
    valid_to
)
SELECT
    s.policy_nk,
    s.category,
    s.criticality,
    s.quarantine,
    s.added_by,
    TRUE,
    current_timestamp(),
    NULL
FROM prepared_run_policy s
LEFT ANTI JOIN {catalog}.{schema}.run_policy t
    ON  t.policy_nk = s.policy_nk
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.category AS string), '_NULL_'),
                coalesce(cast(t.criticality AS string), '_NULL_'),
                coalesce(cast(t.quarantine AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared_policy)
    spark.sql(sql_close_policy)
    spark.sql(sql_close_missing_policy)
    spark.sql(sql_insert_policy)
    print("SCD2 run_policy update completed successfully.")
except Exception as e:
    print("Error:", e)

##### rule_template ######

df_templates = spark.createDataFrame(user_values_rule_templates)
df_templates.createOrReplaceTempView("templates_inputs")

sql_prepared_rt = f"""
CREATE OR REPLACE TEMP VIEW prepared_rule_templates AS
SELECT
    src.name AS template_nk,
    src.name AS name,
    src.description,
    rt.rule_type_id,
    rd.rule_dimension_id,
    src.scope,
    src.is_reusable,
    src.engine_type,
    src.statement,
    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(rt.rule_type_id AS string), '_NULL_'),
            coalesce(cast(rd.rule_dimension_id AS string), '_NULL_'),
            coalesce(cast(src.scope AS string), '_NULL_'),
            coalesce(cast(src.is_reusable AS string), '_NULL_')
        ), 256
    ) AS new_hash

FROM templates_inputs src
JOIN {catalog}.{schema_metadata}.rule_type rt
    ON rt.rule_type_nk = src.rule_type_nk
    AND rt.is_active = TRUE
JOIN {catalog}.{schema_metadata}.rule_dimension rd
    ON rd.rule_dimension_nk = src.rule_dimension_nk
    AND rd.is_active = TRUE
"""

sql_close_previous_rt = f"""
MERGE INTO {catalog}.{schema}.rule_template AS target
USING prepared_rule_templates AS source
ON target.template_nk = source.template_nk
AND target.is_active = TRUE
AND sha2(
        concat_ws(
            '||',
            coalesce(cast(target.rule_type_id AS string), '_NULL_'),
            coalesce(cast(target.rule_dimension_id AS string), '_NULL_'),
            coalesce(cast(target.scope AS string), '_NULL_'),
            coalesce(cast(target.is_reusable AS string), '_NULL_')
        ), 256
    ) <> source.new_hash

WHEN MATCHED THEN
UPDATE SET
    target.is_active = FALSE,
    target.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing_rt = f"""
UPDATE {catalog}.{schema}.rule_template t
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_rule_templates s
    WHERE s.template_nk = t.template_nk
)
"""

sql_insert_new_rt = f"""
INSERT INTO {catalog}.{schema}.rule_template (
    name,
    template_nk,
    description,
    rule_type_id,
    rule_dimension_id,
    is_reusable,
    scope,
    engine_type,
    statement,
    added_by,
    is_active,
    valid_from,
    valid_to
)
SELECT
    s.name,
    s.template_nk,
    s.description,
    s.rule_type_id,
    s.rule_dimension_id,
    s.is_reusable,
    s.scope,
    s.engine_type,
    s.statement,
    s.added_by,
    TRUE,
    current_timestamp(),
    NULL
FROM prepared_rule_templates s
LEFT ANTI JOIN {catalog}.{schema}.rule_template t
    ON  t.template_nk = s.template_nk
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.rule_type_id AS string), '_NULL_'),
                coalesce(cast(t.rule_dimension_id AS string), '_NULL_'),
                coalesce(cast(t.scope AS string), '_NULL_'),
                coalesce(cast(t.is_reusable AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared_rt)
    spark.sql(sql_close_previous_rt)
    spark.sql(sql_close_missing_rt)
    spark.sql(sql_insert_new_rt)
    print("SCD2 rule_template update completed successfully.")
except Exception as e:
    print("Error:", e)

##### apply_at ######

df_apply_at = spark.createDataFrame(user_values_apply_at)
df_apply_at.createOrReplaceTempView("apply_at_inputs")

sql_prepared_apply_at = f"""
CREATE OR REPLACE TEMP VIEW prepared_apply_at AS
SELECT
    src.apply_at,
    src.is_project_specific,
    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(src.apply_at AS string), '_NULL_'),
            coalesce(cast(src.is_project_specific AS string), '_NULL_')
        ), 256
    ) AS new_hash
FROM apply_at_inputs src
"""

sql_close_previous_apply_at = f"""
MERGE INTO {catalog}.{schema}.apply_at AS target
USING prepared_apply_at AS source
ON target.apply_at = source.apply_at
AND target.is_active = TRUE
AND sha2(
        concat_ws(
            '||',
            coalesce(cast(target.apply_at AS string), '_NULL_'),
            coalesce(cast(target.is_project_specific AS string), '_NULL_')
        ), 256
    ) <> source.new_hash

WHEN MATCHED THEN
UPDATE SET
    target.is_active = FALSE,
    target.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing_apply_at = f"""
UPDATE {catalog}.{schema}.apply_at t
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_apply_at s
    WHERE s.apply_at = t.apply_at
)
"""

sql_insert_new_apply_at = f"""
INSERT INTO {catalog}.{schema}.apply_at (
    apply_at,
    is_project_specific,
    is_active,
    added_by,
    valid_from,
    valid_to
)
SELECT
    s.apply_at,
    s.is_project_specific,
    TRUE,
    s.added_by,
    current_timestamp(),
    NULL
FROM prepared_apply_at s
LEFT ANTI JOIN {catalog}.{schema}.apply_at t
    ON  t.apply_at = s.apply_at
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.apply_at AS string), '_NULL_'),
                coalesce(cast(t.is_project_specific AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared_apply_at)
    spark.sql(sql_close_previous_apply_at)
    spark.sql(sql_close_missing_apply_at)
    spark.sql(sql_insert_new_apply_at)
    print("SCD2 apply_at update completed successfully.")
except Exception as e:
    print("Error:", e)

##### rule_assingment ######

df_rules = spark.createDataFrame(user_values_rule_assignment)
df_rules.createOrReplaceTempView("rules_assignment_inputs")

sql_prepared_ra = f"""
CREATE OR REPLACE TEMP VIEW prepared_rule_assignment AS
SELECT
    b.rule_template_id,
    src.template_nk,
    c.policy_id,
    a.table_id,
    src.table_nk,
    proj.project_id,
    NULLIF(src.project_nk, '') AS project_nk,
    e.apply_at_id,
    src.apply_at_nk,

    NULLIF(src.parameters_identifiers, '') AS parameters_identifiers,
    NULLIF(src.parameters_values, '') AS parameters_values,

    CONCAT(
        src.template_nk, '.',
        COALESCE(NULLIF(src.project_nk, ''), '_NULL_'), '.',
        src.table_nk, '.',
        src.apply_at_nk, '.',
        COALESCE(
            to_json(
                from_json(
                    NULLIF(src.parameters_identifiers, ''),
                    'MAP<STRING, STRING>'
                )
            ),
            '_NULL_'
        )
    ) AS rule_assignment_nk,

    src.added_by,

    sha2(
        concat_ws(
            '||',
            coalesce(cast(b.rule_template_id AS string), '_NULL_'),
            coalesce(cast(c.policy_id AS string), '_NULL_'),
            coalesce(cast(a.table_id AS string), '_NULL_'),
            coalesce(cast(proj.project_id AS string), '_NULL_'),
            coalesce(cast(e.apply_at_id AS string), '_NULL_'),
            coalesce(cast(NULLIF(src.parameters_values, '') AS string), '_NULL_')
        ), 256
    ) AS new_hash

FROM rules_assignment_inputs src
JOIN {catalog}.{schema_metadata}.table a
    ON a.table_nk = src.table_nk 
    AND a.is_active = TRUE
JOIN {catalog}.{schema}.rule_template b
    ON b.template_nk = src.template_nk 
    AND b.is_active = TRUE
JOIN {catalog}.{schema}.run_policy c
    ON c.policy_nk = src.policy_nk 
    AND c.is_active = TRUE
JOIN {catalog}.{schema}.apply_at e
    ON e.apply_at_nk = src.apply_at_nk 
    AND e.is_active = TRUE
LEFT JOIN {catalog}.{schema_metadata}.project proj
    ON proj.project_nk = src.project_nk
    AND proj.is_active = TRUE
"""

sql_close_ra = f"""
MERGE INTO {catalog}.{schema}.rule_assignment AS t
USING prepared_rule_assignment AS s
ON t.rule_assignment_nk = s.rule_assignment_nk
AND t.is_active = TRUE
AND sha2(
        concat_ws(
            '||',
            coalesce(cast(t.rule_template_id AS string), '_NULL_'),
            coalesce(cast(t.policy_id AS string), '_NULL_'),
            coalesce(cast(t.table_id AS string), '_NULL_'),
            coalesce(cast(t.project_id AS string), '_NULL_'),
            coalesce(cast(t.apply_at_id AS string), '_NULL_'),
            coalesce(cast(NULLIF(t.parameters_values, '') AS string), '_NULL_')
        ), 256
    ) <> s.new_hash

WHEN MATCHED THEN
UPDATE SET
    t.is_active = FALSE,
    t.valid_to = CURRENT_TIMESTAMP()
"""

sql_close_missing_ra = f"""
UPDATE {catalog}.{schema}.rule_assignment t
SET
    t.valid_to = current_timestamp(),
    t.is_active = false
WHERE t.valid_to IS NULL
AND NOT EXISTS (
    SELECT 1
    FROM prepared_rule_assignment s
    WHERE s.rule_assignment_nk = t.rule_assignment_nk
)
"""

sql_insert_ra = f"""
INSERT INTO {catalog}.{schema}.rule_assignment (
    rule_template_id,
    template_nk,
    policy_id,
    table_id,
    table_nk,
    project_id,
    project_nk,
    apply_at_id,
    apply_at_nk,
    parameters_identifiers,
    parameters_values,
    added_by,
    is_active,
    valid_from,
    valid_to
)
SELECT
    s.rule_template_id,
    s.template_nk,
    s.policy_id,
    s.table_id,
    s.table_nk,
    s.project_id,
    s.project_nk,
    s.apply_at_id,
    s.apply_at_nk,
    s.parameters_identifiers,
    s.parameters_values,
    s.added_by,
    TRUE,
    current_timestamp(),
    NULL
FROM prepared_rule_assignment s
LEFT ANTI JOIN {catalog}.{schema}.rule_assignment t
    ON  t.rule_assignment_nk = s.rule_assignment_nk
    AND t.is_active = TRUE
    AND sha2(
            concat_ws(
                '||',
                coalesce(cast(t.rule_template_id AS string), '_NULL_'),
                coalesce(cast(t.policy_id AS string), '_NULL_'),
                coalesce(cast(t.table_id AS string), '_NULL_'),
                coalesce(cast(t.project_id AS string), '_NULL_'),
                coalesce(cast(t.apply_at_id AS string), '_NULL_'),
                coalesce(cast(NULLIF(t.parameters_values, '') AS string), '_NULL_')
            ), 256
        ) = s.new_hash
"""

try:
    spark.sql(sql_prepared_ra)
    spark.sql(sql_close_ra)
    spark.sql(sql_close_missing_ra)
    spark.sql(sql_insert_ra)
    print("SCD2 rule_assignment update completed successfully.")
except Exception as e:
    print("Error:", e)